# 03. Denoising con Reglas (Core)

**Objetivo:** remover ruido administrativo y conservar señal clínica con lógica de aseveración validada.
**Entradas (inputs):** `data/splits/dataset_base.csv` y `data/splits/*_indices.csv`.
**Salidas (outputs):** `data/splits/train_denoised.csv`, `dev_denoised.csv`, `test_denoised.csv`, `data/dataset_denoised.csv`, `data/input_for_gemini.json`.
**Notebook anterior:** `notebooks/pipeline/02_patient_level_split.ipynb`.
**Notebook siguiente:** `notebooks/pipeline/04a_linea_base_dummy.ipynb`, `notebooks/pipeline/04b_linea_base_tfidf.ipynb`, `notebooks/pipeline/04c_linea_base_transformers.ipynb`.


## Técnicas, herramientas y librerías de esta etapa

- **Técnica principal:** denoising clínico rule-based con filtro de aseveración clínica.
- **Herramientas/librerías:** fork `Spanish_Psych_Phenotyping_PY`, `spaCy`, componentes de `medSpaCy`, matcher de reglas clínicas del fork y `utils_shared.keep_entity` como política centralizada de conservación/descarte.
- **Por qué es adecuada aquí:** el problema real del corpus no es solo clasificación, sino mezcla de notas diagnósticas con seguimiento, plantillas y negaciones no atribuibles al paciente. Para ese tipo de ruido documental, un filtro explícito y auditable es más defendible que una limpieza implícita.
- **Limitación:** no recupera bien fenómenos semánticos fuera del alcance del léxico/reglas y depende de la calidad del matcher clínico.
- **Alternativa si el objetivo fuera otro:** modelos aprendidos de NER + assertion status podrían capturar más variación libre, pero hoy serían menos auditables y abrirían una capa metodológica nueva que no conviene introducir antes de `test`.


In [ ]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# Importar utilidades compartidas
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    sys.path.append(str(current_dir.parent))
else:
    sys.path.append(str(current_dir))

try:
    from notebooks.utils_shared import setup_paths, load_splits, keep_entity
except ImportError:
    sys.path.append(str(current_dir))
    from utils_shared import setup_paths, load_splits, keep_entity

paths = setup_paths()
DATA_PATH = paths['DATA_PATH']
SPLITS_PATH = paths['SPLITS_PATH']
FORK_PATH = paths['FORK_PATH']

# Importar utilidades CLI del fork
sys.path.insert(0, str(FORK_PATH))
try:
    from cli import build_pipeline, load_yaml
    print("CLI importado correctamente")
except ImportError:
    print("Error: No se pudo importar cli.py. Verifica FORK_PATH.")
    raise

print(f"Rutas configuradas. Splits en: {SPLITS_PATH}")

## 1. Cargar Datos y Pipeline NLP


In [ ]:
# Cargar particiones
df_base, train_idx, dev_idx, test_idx = load_splits(SPLITS_PATH)
print(f"Dataset total: {len(df_base)} casos")

# Cargar flujo NLP clínico
print("Cargando pipeline NLP...")
config_path = FORK_PATH / "configs" / "fenotipos.yml"
fenos_cfg = load_yaml(config_path)
nlp = build_pipeline("py", fenos_cfg)
print("Pipeline cargado.")

## 2. Ejecutar Denoising (Detección de Señal)


### Justificación: por qué el Denoising se ejecuta con el perfil **core** (y se aplica a todo el dataset)

En EHR reales (como IPS) aparecen **segmentos repetitivos de formulario** y negaciones de tipo checklist (*“Sin síntomas”, “Examen normal”*).
Si esos fragmentos se tratan como evidencia clínica, inflan falsos positivos en extracción basada en reglas.

En este proyecto, el denoising es **normalización del formato de nota** (*data cleaning*), no una “decisión clínica”:
- Eliminamos/ignoramos menciones **históricas/hipotéticas/familiares**.
- Tratamos negaciones del **médico/plantilla** como ruido.
- Conservamos negaciones explícitas del **paciente** (*“Paciente niega…”*) como señal fenomenológica.

Para que el pipeline sea reproducible y la ablación sea justa:
- **03** se ejecuta una sola vez con `core` para producir un dataset “denoised” único.
- Luego, **07** extrae features con `core` y `py` **sobre el mismo dataset denoised**.


In [ ]:
# Construir flujo NLP (perfil core) y ejecutar denoising con `nlp.pipe`
# Importante: este notebook define el conjunto de datos depurado universal.
# La lógica de aseveración clínica (`keep_entity`) debe mantenerse idéntica en 03/06/08.
# Criterio clínico: negación del paciente se conserva como señal; negación de plantilla/médico se filtra como ruido.

FENOS_CFG = load_yaml(FORK_PATH / 'configs' / 'fenotipos.yml')
nlp = build_pipeline('core', FENOS_CFG)

print("-> Ejecutando denoising con nlp.pipe...")

texts = df_base['texto'].fillna('').astype(str).tolist()

has_signal = []
feat_negacion_paciente = []

for doc in tqdm(nlp.pipe(texts, batch_size=128, n_process=1), total=len(texts)):
    any_kept = False
    any_patient_neg = False

    for ent in getattr(doc, "ents", []):
        keep, is_pat_neg = keep_entity(ent, doc, window_tokens=12)
        if keep:
            any_kept = True
        if is_pat_neg:
            any_patient_neg = True
        if any_kept and any_patient_neg:
            break

    has_signal.append(any_kept)
    feat_negacion_paciente.append(int(any_patient_neg))

df_base['has_clinical_signal'] = has_signal
df_base['feat_negacion_paciente'] = feat_negacion_paciente

print(f"Denoising finalizado. has_clinical_signal=True en {sum(has_signal)}/{len(has_signal)} "
      f"({sum(has_signal)/len(has_signal):.1%}) notas.")


## 3. Filtrar Train Set


In [ ]:
# ============================
# Exportación alineada al flujo
# ============================
# - Mantener nombres de particiones: train_denoised / dev_denoised / test_denoised
# - Exportar conjunto de datos completo con banderas (para auditoría)
# - Exportar conjunto de datos depurado completo (insumo para 06 y caché LLM)

def _export_split(name: str, idx):
    mask = df_base['row_id'].isin(idx)
    cases = df_base[mask]
    kept = cases[cases['has_clinical_signal']]
    removed = cases[~cases['has_clinical_signal']]

    print(f"{name.upper()} original: {len(cases)}")
    print(f"{name.upper()} denoised: {len(kept)} ({len(kept)/max(1,len(cases)):.1%})")
    print(f"{name.upper()} eliminados (ruido): {len(removed)}")

    out = SPLITS_PATH / f"{name}_denoised.csv"
    kept.to_csv(out, index=False)
    print(f"Guardado: {out}")

    return kept, removed

train_kept, train_removed = _export_split('train', train_idx)
dev_kept, dev_removed     = _export_split('dev',   dev_idx)
test_kept, test_removed   = _export_split('test',  test_idx)

# Conjunto de datos completo con banderas (útil para análisis y auditoría)
flags_path = DATA_PATH / "dataset_with_clinical_signal_flag.csv"
df_base.to_csv(flags_path, index=False)
print(f"Guardado dataset con flags: {flags_path}")

# Conjunto de datos depurado completo (insumo para caché LLM y notebook 06)
# Nota: este archivo define el universo de notas válidas post-denoising.
denoised_full = df_base[df_base['has_clinical_signal']].copy()
denoised_path = DATA_PATH / "dataset_denoised.csv"
denoised_full.to_csv(denoised_path, index=False)
print(f"Guardado dataset denoised completo: {denoised_path} | rows={len(denoised_full)}")


## 4. Análisis de lo Eliminado
Verificar qué estamos eliminando para asegurar que sea ruido.


In [ ]:
if len(train_removed) > 0:
    print("Ejemplos de textos eliminados (sin señal clínica detectada):")
    print("-"*60)
    for txt in train_removed['texto'].head(5):
        print(f"• {txt[:150]}...")
else:
    print("No se eliminó ningún caso.")

In [ ]:
# =====================================
# Generar entrada para LLM (Gemini)
# =====================================

gemini_input = df_base[df_base["has_clinical_signal"]].copy()

gemini_payload = [
    {
        "row_id": int(r["row_id"]),
        "texto": r["texto"]
    }
    for _, r in gemini_input.iterrows()
]

output_path = DATA_PATH / "input_for_gemini.json"

import json
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(gemini_payload, f, ensure_ascii=False, indent=2)

print(f"Gemini input guardado en {output_path}")

## Resumen reutilizable del filtrado para validación clínica

Este notebook ya define el universo post-denoising, pero conviene leerlo con tres aclaraciones metodológicas explícitas:

- la plantilla administrativa se **limpia dentro de la nota**; no se elimina automáticamente una consulta solo por tener bloque plantilla;
- el descarte ocurre cuando, después de la limpieza y de la lógica de aseveración, **no queda señal clínica útil**;
- la revisión clínica externa puede reutilizar estos artefactos sin recalcular el pipeline.


In [ ]:
# Resumen rápido del filtrado para auditoría clínica
resumen_filtrado_03 = pd.DataFrame([
    {
        'etapa': 'dataset_base',
        'n_registros': len(df_base),
        'n_pacientes': df_base['patient_id'].nunique(),
        'comentario': 'Notas post-deduplicación y post-limpieza de plantilla dentro de la nota.'
    },
    {
        'etapa': 'sin_senal_clinica_util',
        'n_registros': int((~df_base['has_clinical_signal']).sum()),
        'n_pacientes': df_base.loc[~df_base['has_clinical_signal'], 'patient_id'].nunique(),
        'comentario': 'Consultas descartadas tras denoising por ausencia de fenomenología útil.'
    },
    {
        'etapa': 'dataset_denoised_final',
        'n_registros': len(denoised_full),
        'n_pacientes': denoised_full['patient_id'].nunique(),
        'comentario': 'Universo final consumido por 04-10.'
    },
    {
        'etapa': 'train/dev/test_denoised',
        'n_registros': f"{len(train_kept)}/{len(dev_kept)}/{len(test_kept)}",
        'n_pacientes': f"{train_kept['patient_id'].nunique()}/{dev_kept['patient_id'].nunique()}/{test_kept['patient_id'].nunique()}",
        'comentario': 'Tamaños finales por split tras filtrado clínico.'
    },
])
display(resumen_filtrado_03)
